# Notebook 02 — Probability Averaging and Cross-Validation Ensembling

**Session:** Ensembling  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Implement arithmetic, geometric, and harmonic probability averaging across heterogeneous estimators (SVC, Random Forest, KNeighborsClassifier) within a disciplined 5-fold cross-validation pipeline, collecting both Out-of-Fold (OOF) validation metrics and test-set ensemble predictions.

> **Mathematical Foundation:** Averaging independent probability estimates shrinks prediction variance according to $\text{Var}(\bar{f}) = \frac{1}{M}\overline{\text{Var}} + \frac{M-1}{M}\overline{\text{Cov}}$, directly reducing expected cross-entropy and Mean Squared Error.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import gmean, hmean
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## 1. Load Dataset & Train-Test Split

We load `synthetic_classification.csv` from `../data/`, containing 20 features and binary ground truth targets.

In [ ]:
data_path = Path("../data") / "synthetic_classification.csv"
df = pd.read_csv(data_path)
X = df.drop(columns=["target"]).values
y = df["target"].values

# Partition into 80% train and 20% held-out test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

print(f"X_train_full shape: {X_train_full.shape} | X_test shape: {X_test.shape}")
print(f"Train positive class ratio: {y_train_full.mean():.3f} | Test: {y_test.mean():.3f}")

## 2. Define Probability Averaging Schemes

We define:
1. **Arithmetic Mean:** $\bar{p}_{\text{arith}} = \frac{1}{M} \sum_{m=1}^M p_m$
2. **Geometric Mean:** $\bar{p}_{\text{geom}} \propto \left(\prod_{m=1}^M p_m\right)^{1/M}$
3. **Harmonic Mean:** $\bar{p}_{\text{harm}} = \frac{M}{\sum_{m=1}^M \frac{1}{p_m}}$

In [ ]:
def ensemble_arithmetic(proba_matrix: np.ndarray) -> np.ndarray:
    """Compute simple arithmetic average of predicted probabilities across columns."""
    return proba_matrix.mean(axis=1)


def ensemble_geometric(proba_matrix: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Compute geometric mean of predicted probabilities across columns."""
    clipped = np.clip(proba_matrix, eps, 1.0 - eps)
    return gmean(clipped, axis=1)


def ensemble_harmonic(proba_matrix: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Compute harmonic mean of predicted probabilities across columns."""
    clipped = np.clip(proba_matrix, eps, 1.0 - eps)
    return hmean(clipped, axis=1)

## 3. Stratified 5-Fold Cross-Validation Pipeline

We initialize 3 diverse base classifiers:
- `SVC(probability=True)`: Margin-based maximum separation
- `RandomForestClassifier`: Ensemble of randomized decision trees
- `KNeighborsClassifier`: Local manifold distance-based estimation

On each fold, we record Out-Of-Fold (OOF) validation predictions and generate fold test predictions.

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_1 = SVC(probability=True, kernel="rbf", random_state=42)
model_2 = RandomForestClassifier(n_estimators=100, random_state=42)
model_3 = KNeighborsClassifier(n_neighbors=7)

oof_m1 = np.zeros(len(y_train_full))
oof_m2 = np.zeros(len(y_train_full))
oof_m3 = np.zeros(len(y_train_full))

test_preds_m1 = []
test_preds_m2 = []
test_preds_m3 = []

fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_full, y_train_full)):
    X_tr, y_tr = X_train_full[train_idx], y_train_full[train_idx]
    X_val, y_val = X_train_full[val_idx], y_train_full[val_idx]

    # Fit base models
    model_1.fit(X_tr, y_tr)
    model_2.fit(X_tr, y_tr)
    model_3.fit(X_tr, y_tr)

    # Predict validation fold
    p1_val = model_1.predict_proba(X_val)[:, 1]
    p2_val = model_2.predict_proba(X_val)[:, 1]
    p3_val = model_3.predict_proba(X_val)[:, 1]

    oof_m1[val_idx] = p1_val
    oof_m2[val_idx] = p2_val
    oof_m3[val_idx] = p3_val

    # Predict test set
    test_preds_m1.append(model_1.predict_proba(X_test)[:, 1])
    test_preds_m2.append(model_2.predict_proba(X_test)[:, 1])
    test_preds_m3.append(model_3.predict_proba(X_test)[:, 1])

    val_matrix = np.column_stack([p1_val, p2_val, p3_val])
    p_arith_val = ensemble_arithmetic(val_matrix)
    p_geom_val = ensemble_geometric(val_matrix)
    p_harm_val = ensemble_harmonic(val_matrix)

    fold_results.append(
        {
            "Fold": fold + 1,
            "SVC_AUC": roc_auc_score(y_val, p1_val),
            "RF_AUC": roc_auc_score(y_val, p2_val),
            "KNN_AUC": roc_auc_score(y_val, p3_val),
            "Arith_AUC": roc_auc_score(y_val, p_arith_val),
            "Geom_AUC": roc_auc_score(y_val, p_geom_val),
            "Harm_AUC": roc_auc_score(y_val, p_harm_val),
        }
    )

fold_df = pd.DataFrame(fold_results)
print("=== Per-Fold ROC-AUC Performance ===")
print(fold_df.round(4).to_string(index=False))

## 4. Full Out-of-Fold (OOF) & Test Set Performance Evaluation

We evaluate the full OOF validation score and fold-averaged test predictions.

In [ ]:
# Construct full OOF prediction matrix
oof_matrix = np.column_stack([oof_m1, oof_m2, oof_m3])
oof_arith = ensemble_arithmetic(oof_matrix)
oof_geom = ensemble_geometric(oof_matrix)
oof_harm = ensemble_harmonic(oof_matrix)

print("=== Full Out-of-Fold (OOF) ROC-AUC Scores ===")
print(f"SVC OOF AUC:            {roc_auc_score(y_train_full, oof_m1):.4f}")
print(f"Random Forest OOF AUC:  {roc_auc_score(y_train_full, oof_m2):.4f}")
print(f"KNN OOF AUC:            {roc_auc_score(y_train_full, oof_m3):.4f}")
print(f"Arithmetic Ensemble:    {roc_auc_score(y_train_full, oof_arith):.4f}")
print(f"Geometric Ensemble:     {roc_auc_score(y_train_full, oof_geom):.4f}")
print(f"Harmonic Ensemble:      {roc_auc_score(y_train_full, oof_harm):.4f}")

# Average test predictions across the 5 folds
avg_test_m1 = np.mean(test_preds_m1, axis=0)
avg_test_m2 = np.mean(test_preds_m2, axis=0)
avg_test_m3 = np.mean(test_preds_m3, axis=0)

test_matrix = np.column_stack([avg_test_m1, avg_test_m2, avg_test_m3])
test_arith = ensemble_arithmetic(test_matrix)

print("\n=== Independent Test Set ROC-AUC Scores ===")
print(f"SVC Test AUC:           {roc_auc_score(y_test, avg_test_m1):.4f}")
print(f"Random Forest Test AUC: {roc_auc_score(y_test, avg_test_m2):.4f}")
print(f"KNN Test AUC:           {roc_auc_score(y_test, avg_test_m3):.4f}")
print(f"Arithmetic Ensemble:    {roc_auc_score(y_test, test_arith):.4f}")